# Geotechnical Data Analysis
**Targets:** `Phi_deg` (friction angle φ) and `Cu_kPa` (undrained cohesion Cu)

This notebook covers:
1. DataFrame loading and exploration
2. Empirical estimation of missing Mv values
3. Feature engineering and variation
4. SPT → CPT conversion
5. Linear correlation analysis (feature impact)

---

## 1. Load data into DataFrame

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

In [ ]:
# ── Load the merged Excel file ──────────────────────────────────────────────
FILE = 'geotechnical_data_MERGED.xlsx'   # adjust path if needed

df = pd.read_excel(FILE, sheet_name='All_Geotechnical_Data', index_col='Index')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Column types and null counts ────────────────────────────────────────────
info = pd.DataFrame({
    'dtype':    df.dtypes,
    'non_null': df.notna().sum(),
    'null':     df.isna().sum(),
    'pct_fill': (df.notna().sum() / len(df) * 100).round(1)
})
info

In [ ]:
# ── Completeness bar chart ──────────────────────────────────────────────────
pct = (df.notna().sum() / len(df) * 100).sort_values()
colors = ['#e24b4a' if v == 0 else '#ef9f27' if v < 60 else '#3266ad' if v < 100 else '#1d9e75'
          for v in pct]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(pct.index, pct.values, color=colors, height=0.6)
ax.set_xlabel('% complete')
ax.set_xlim(0, 110)
for bar, v in zip(bars, pct.values):
    ax.text(v + 1, bar.get_y() + bar.get_height()/2,
            f'{v:.0f}%', va='center', fontsize=9)
patches = [mpatches.Patch(color=c, label=l) for c, l in
           [('#1d9e75','100%'), ('#3266ad','60–99%'), ('#ef9f27','<60%'), ('#e24b4a','0%')]]
ax.legend(handles=patches, fontsize=9, loc='lower right')
ax.set_title('Column completeness across 240 records')
plt.tight_layout()
plt.show()

In [ ]:
# ── Descriptive statistics ──────────────────────────────────────────────────
df.describe().round(3)

---
## 2. Empirical estimation of coefficient of compressibility (Mv)

Several empirical routes exist to estimate or fill missing Mv values from Atterberg limits:

| Formula | Source | Expression |
|---|---|---|
| Skempton & Jones | 1944 | `mv ≈ 0.0001 × (LL − 10)` |
| Terzaghi Cc → mv | 1943 | `Cc = 0.009×(LL−10)`, then `mv = Cc / [2.303×(1+e₀)×σ']` |
| Mayne & Kulhawy PI | 1982 | `mv ≈ 0.00168 + 0.00033×PI` |
| Stroud SPT | 1974 | `E' = f₁×N`, `mv ≈ 1/(E'×1000)` |

In [ ]:
# ── Empirical Mv estimates ───────────────────────────────────────────────────
df['Mv_SJ']  = 0.0001 * (df['LL'] - 10)                    # Skempton & Jones
df['Mv_MK']  = 0.00168 + 0.00033 * df['PI']               # Mayne & Kulhawy
df['Cc_TZ']  = 0.009  * (df['LL'] - 10)                    # Terzaghi Cc
df['Cc_REM'] = 0.007  * (df['LL'] - 10)                    # Remoulded Cc

# Fill missing Mv_50kPa with Skempton & Jones estimate where absent
df['Mv_50kPa_filled'] = df['Mv_50kPa'].fillna(df['Mv_SJ'])

# Approximate Mv_200 where missing: Mv_200 ≈ 0.60 × Mv_50 (NC clay ratio)
df['Mv_200kPa_filled'] = df['Mv_200kPa'].fillna(df['Mv_50kPa_filled'] * 0.60)

print('Mv_50kPa  — original null count:', df['Mv_50kPa'].isna().sum())
print('Mv_50kPa  — filled   null count:', df['Mv_50kPa_filled'].isna().sum())
print('Mv_200kPa — original null count:', df['Mv_200kPa'].isna().sum())
print('Mv_200kPa — filled   null count:', df['Mv_200kPa_filled'].isna().sum())

In [ ]:
# ── Compare measured vs empirical Mv_50 ────────────────────────────────────
mask = df['Mv_50kPa'].notna()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scatter: measured vs Skempton & Jones
ax = axes[0]
ax.scatter(df.loc[mask, 'Mv_50kPa'], df.loc[mask, 'Mv_SJ'],
           alpha=0.5, s=30, color='#3266ad', edgecolors='none')
lim = [0, df.loc[mask, ['Mv_50kPa','Mv_SJ']].max().max() * 1.05]
ax.plot(lim, lim, 'r--', linewidth=1, label='1:1 line')
r, p = stats.pearsonr(df.loc[mask, 'Mv_50kPa'], df.loc[mask, 'Mv_SJ'])
ax.set_title(f'Measured vs Skempton & Jones  (r = {r:.3f})')
ax.set_xlabel('Measured Mv_50 (m²/MN×10⁻⁴)')
ax.set_ylabel('Empirical Mv_SJ')
ax.legend(fontsize=9)

# Scatter: measured vs Mayne & Kulhawy
ax = axes[1]
ax.scatter(df.loc[mask, 'Mv_50kPa'], df.loc[mask, 'Mv_MK'],
           alpha=0.5, s=30, color='#1d9e75', edgecolors='none')
ax.plot(lim, lim, 'r--', linewidth=1, label='1:1 line')
r2, _ = stats.pearsonr(df.loc[mask, 'Mv_50kPa'], df.loc[mask, 'Mv_MK'])
ax.set_title(f'Measured vs Mayne & Kulhawy  (r = {r2:.3f})')
ax.set_xlabel('Measured Mv_50 (m²/MN×10⁻⁴)')
ax.set_ylabel('Empirical Mv_MK')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 3. Feature engineering and variation

In [ ]:
# ── Derived features ────────────────────────────────────────────────────────

# Liquidity Index: position of Wn relative to plastic range
df['LI'] = (df['Wn_%'] - df['PL']) / df['PI'].replace(0, np.nan)

# Grain size gradient: difference between coarser and finer fraction
df['Grain_coarse'] = df['Grain_2.36mm'].fillna(df['Grain_2.00mm'])   # unify sieves
df['DeltaGrain']   = df['Grain_0.425mm'] - df['Grain_0.075mm']

# Compression ratio (non-linearity of compressibility)
df['Mv_ratio'] = df['Mv_50kPa_filled'] / df['Mv_200kPa_filled'].replace(0, np.nan)

# Cv/Mv drainage index (proxy for hydraulic conductivity)
df['Drain_index'] = df['Cv_50kPa'] / df['Mv_50kPa_filled'].replace(0, np.nan)

# Normalised SPT (approximate, assuming σ'v ≈ 18×depth, depth not stored — use raw)
df['SPT_norm'] = df['SPT']

print('New features added:', ['LI','Grain_coarse','DeltaGrain','Mv_ratio','Drain_index'])
df[['LL','PI','LI','DeltaGrain','Mv_ratio','Drain_index']].describe().round(3)

In [ ]:
# ── Feature distribution plots ──────────────────────────────────────────────
feature_cols = ['LL','PI','Sat_Unit_Wt_kN_m3','Mv_50kPa','Cv_50kPa',
                'SPT','Grain_0.425mm','Grain_0.075mm']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, col in zip(axes, feature_cols):
    data = df[col].dropna()
    ax.hist(data, bins=25, color='#3266ad', alpha=0.75, edgecolor='none')
    ax.axvline(data.mean(), color='#e24b4a', linewidth=1.2,
               linestyle='--', label=f'mean={data.mean():.1f}')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature distributions (n=240)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pairplot — key features vs targets ─────────────────────────────────────
plot_cols = ['LL','PI','Sat_Unit_Wt_kN_m3','Mv_50kPa','Phi_deg','Cu_kPa']
pair_df = df[plot_cols].dropna()

g = sns.pairplot(pair_df, diag_kind='kde', plot_kws={'alpha':0.4, 's':15},
                 diag_kws={'color':'#3266ad'})
g.fig.suptitle('Pairwise feature relationships', y=1.01)
plt.show()

---
## 4. SPT → CPT conversion

Your dataset has **0 CPT values** and **122 SPT values**. We derive CPT from SPT using Robertson & Campanella (1983):

```
qc (MPa) = (qc/N60) × N60
```

Typical `qc/N60` ratios by soil group:

| Soil type | USC class | qc/N₆₀ |
|---|---|---|
| Gravel/sand | SW, SP, GW | 0.40–0.60 |
| Silty sand | SM, SC-SM | 0.30–0.40 |
| Sandy silt/clay | SC, CL | 0.20–0.30 |
| Clay | CH, OH, MH | 0.10–0.20 |

In [ ]:
# ── SPT → CPT conversion ────────────────────────────────────────────────────
# Your SPT column is in kPa (bearing capacity), not raw N-value.
# Convert: N60 ≈ SPT_kPa / 10  (approximate — confirm with supervisor)
df['N60'] = df['SPT'] / 10.0

# Assign qc/N ratio based on PI (proxy for soil type when USC class not available)
def qc_n_ratio(pi):
    """Robertson & Campanella ratio from plasticity index."""
    if pd.isna(pi):   return 0.35   # default silty sand
    if pi == 0:       return 0.50   # clean sand (NP)
    if pi <= 10:      return 0.35   # silty sand / SC-SM
    if pi <= 20:      return 0.25   # sandy silt / SC / CL
    return 0.15                     # plastic clay CH/OH

df['qc_N_ratio'] = df['PI'].apply(qc_n_ratio)
df['CPT_est_MPa'] = df['N60'] * df['qc_N_ratio']

# Fill the original (empty) CPT column
df['CPT_MPa'] = df['CPT_MPa'].fillna(df['CPT_est_MPa'])

print(f"CPT non-null after filling: {df['CPT_MPa'].notna().sum()} / {len(df)}")
df[['SPT','N60','PI','qc_N_ratio','CPT_MPa']].dropna(subset=['SPT']).head(10)

In [ ]:
# ── SPT vs estimated CPT ────────────────────────────────────────────────────
mask = df['SPT'].notna()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scatter by soil group (PI proxy)
ax = axes[0]
groups = [
    (df['PI'] == 0,   '#1d9e75', 'Sand (NP)'),
    ((df['PI'] > 0) & (df['PI'] <= 10), '#3266ad', 'Silty sand (PI≤10)'),
    ((df['PI'] > 10) & (df['PI'] <= 20), '#ef9f27', 'SC/CL (PI 10-20)'),
    (df['PI'] > 20,   '#e24b4a', 'Clay (PI>20)'),
]
for cond, color, label in groups:
    sub = df[mask & cond]
    ax.scatter(sub['SPT'], sub['CPT_MPa'], s=25, alpha=0.6,
               color=color, label=label, edgecolors='none')
ax.set_xlabel('SPT bearing capacity (kPa)')
ax.set_ylabel('Estimated CPT qc (MPa)')
ax.set_title('SPT → CPT by soil group')
ax.legend(fontsize=8)

# Distribution of estimated CPT
ax = axes[1]
df.loc[mask, 'CPT_MPa'].hist(bins=25, ax=ax, color='#3266ad', alpha=0.75, edgecolor='none')
ax.axvline(df.loc[mask, 'CPT_MPa'].mean(), color='#e24b4a', linestyle='--',
           label=f"mean = {df.loc[mask,'CPT_MPa'].mean():.2f} MPa")
ax.set_xlabel('Estimated CPT qc (MPa)')
ax.set_title('Distribution of estimated CPT')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Linear correlation — which features have maximum impact on φ and Cu?

In [ ]:
# ── Full correlation matrix ─────────────────────────────────────────────────
corr_cols = ['LL','PL','PI','Sat_Unit_Wt_kN_m3',
             'Mv_50kPa_filled','Mv_200kPa_filled',
             'Cv_50kPa','Cv_200kPa',
             'SPT','CPT_MPa',
             'Grain_0.425mm','Grain_0.075mm',
             'DeltaGrain','Mv_ratio',
             'Phi_deg','Cu_kPa']

corr_df = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask_upper = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask_upper, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, annot_kws={'size':7})
ax.set_title('Pearson correlation matrix (lower triangle)', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation with targets: ranked bar charts ─────────────────────────────
feature_cols = ['LL','PL','PI','Sat_Unit_Wt_kN_m3',
                'Mv_50kPa_filled','Mv_200kPa_filled',
                'Cv_50kPa','Cv_200kPa',
                'SPT','CPT_MPa',
                'Grain_0.425mm','Grain_0.075mm',
                'DeltaGrain','Mv_ratio']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
targets = [('Phi_deg', 'φ (friction angle)'), ('Cu_kPa', 'Cu (undrained cohesion)')]

for ax, (target, title) in zip(axes, targets):
    corrs = df[feature_cols + [target]].corr()[target].drop(target)
    corrs_sorted = corrs.abs().sort_values(ascending=True)
    signed = corrs[corrs_sorted.index]

    colors = ['#1d9e75' if v > 0 else '#e24b4a' for v in signed]
    bars = ax.barh(signed.index, signed.values, color=colors, height=0.6)

    for bar, val in zip(bars, signed.values):
        xpos = val + 0.01 if val >= 0 else val - 0.01
        ha = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha=ha, fontsize=8)

    ax.axvline(0, color='#888', linewidth=0.8)
    ax.set_xlim(-0.75, 0.75)
    ax.set_title(f'Pearson r with  {title}', fontsize=11)
    ax.set_xlabel('Correlation coefficient r')

    pos_patch = mpatches.Patch(color='#1d9e75', label='positive r')
    neg_patch = mpatches.Patch(color='#e24b4a', label='negative r')
    ax.legend(handles=[pos_patch, neg_patch], fontsize=8)

plt.suptitle('Linear feature correlation with prediction targets', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: top features vs each target ───────────────────────────────────
top_phi = ['Mv_50kPa_filled','Grain_0.425mm','LL','Cv_200kPa']
top_cu  = ['SPT','Grain_0.425mm','Cv_50kPa','Sat_Unit_Wt_kN_m3']

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, feat in enumerate(top_phi):
    ax = axes[0][i]
    sub = df[[feat, 'Phi_deg']].dropna()
    r, _ = stats.pearsonr(sub[feat], sub['Phi_deg'])
    ax.scatter(sub[feat], sub['Phi_deg'], s=18, alpha=0.5,
               color='#3266ad', edgecolors='none')
    m, b = np.polyfit(sub[feat], sub['Phi_deg'], 1)
    xline = np.linspace(sub[feat].min(), sub[feat].max(), 50)
    ax.plot(xline, m*xline + b, color='#e24b4a', linewidth=1.5)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('φ (°)' if i == 0 else '')
    ax.set_title(f'r = {r:.3f}', fontsize=9)

for i, feat in enumerate(top_cu):
    ax = axes[1][i]
    sub = df[[feat, 'Cu_kPa']].dropna()
    r, _ = stats.pearsonr(sub[feat], sub['Cu_kPa'])
    ax.scatter(sub[feat], sub['Cu_kPa'], s=18, alpha=0.5,
               color='#1d9e75', edgecolors='none')
    m, b = np.polyfit(sub[feat], sub['Cu_kPa'], 1)
    xline = np.linspace(sub[feat].min(), sub[feat].max(), 50)
    ax.plot(xline, m*xline + b, color='#e24b4a', linewidth=1.5)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Cu (kPa)' if i == 0 else '')
    ax.set_title(f'r = {r:.3f}', fontsize=9)

axes[0][0].set_ylabel('φ (°)', fontsize=10)
axes[1][0].set_ylabel('Cu (kPa)', fontsize=10)
plt.suptitle('Top features vs targets with linear trend lines', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary: strongest predictors ──────────────────────────────────────────
summary = {}
for target in ['Phi_deg', 'Cu_kPa']:
    corrs = df[feature_cols + [target]].corr()[target].drop(target)
    top3 = corrs.abs().sort_values(ascending=False).head(3)
    summary[target] = [(f, round(corrs[f], 4)) for f in top3.index]

print('=== TOP FEATURES BY PEARSON |r| ===')
print()
for target, feats in summary.items():
    label = 'φ (Phi_deg)' if target == 'Phi_deg' else 'Cu (Cu_kPa)'
    print(f'  {label}:')
    for rank, (feat, r) in enumerate(feats, 1):
        direction = 'positive' if r > 0 else 'negative'
        print(f'    {rank}. {feat:<30s}  r = {r:+.4f}  ({direction})')
    print()

In [ ]:
# ── Final model-ready DataFrame ─────────────────────────────────────────────
model_features = [
    'LL','PL','PI',
    'Sat_Unit_Wt_kN_m3',
    'Mv_50kPa_filled','Mv_200kPa_filled',
    'Cv_50kPa','Cv_200kPa',
    'SPT','CPT_MPa',
    'Grain_0.425mm','Grain_0.075mm',
    'DeltaGrain','Mv_ratio',
    'Phi_deg','Cu_kPa'
]

df_model = df[model_features].copy()

print('Model DataFrame shape:', df_model.shape)
print('Rows with both targets non-null:',
      df_model[['Phi_deg','Cu_kPa']].notna().all(axis=1).sum())
print()
print('Remaining nulls per column:')
print(df_model.isna().sum()[df_model.isna().sum() > 0])
df_model.head()